# Experiment D1: Core Feature Set + Hypertension

This experiment extends Experiment C by adding `CCC_80` (diagnosed high blood pressure) to the literature/core feature set frozen in the controlled experiment setup notebook. It checks whether reported hypertension status adds anything on top of the core 7 variables.

Everything else (population, split, preprocessing style, model, evaluation) is identical to C.

In [1]:
import pandas as pd
import numpy as np

pumf = pd.read_csv("../Data_Données/pumf_cchs.csv")

print("PUMF shape:", pumf.shape)

PUMF shape: (67079, 255)


In [2]:
model_data = pumf[pumf["CCC_05"].isin([1, 2])].copy()

model_data["target"] = (model_data["CCC_05"] == 1).astype(int)

print("Modelling population:", model_data.shape[0])
print(model_data["target"].value_counts().sort_index())

Modelling population: 66242
target
0    60248
1     5994
Name: count, dtype: int64


## Feature set

`D1_Core_Hypertension` from notebook 02: the 7 C features plus `CCC_80`.

In [3]:
FEATURES = [
    "DHHGAGE",
    "DHH_SEX",
    "EDDVH3",
    "BMI_CLASS",
    "INCDGHH",
    "SDCDGIMM",
    "GEOGPRV",
    "CCC_80"
]

print("Number of features:", len(FEATURES))
print(FEATURES)

Number of features: 8
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80']


In [4]:
# Same BMI harmonization as C
model_data["BMI_CLASS"] = np.nan

youth_mask = model_data["DHHGAGE"] == 1
adult_mask = model_data["DHHGAGE"].isin([2, 3, 4, 5])

model_data.loc[youth_mask, "BMI_CLASS"] = model_data.loc[youth_mask, "HWTDGWHO"]
model_data.loc[adult_mask, "BMI_CLASS"] = model_data.loc[adult_mask, "HWTDGISW"]

print(model_data["BMI_CLASS"].value_counts(dropna=False).sort_index())

BMI_CLASS
1.0    26053
2.0    37045
6.0       32
9.0     3112
Name: count, dtype: int64


In [5]:
SPECIAL_CODES = {
    "DHHGAGE": [],
    "DHH_SEX": [],
    "EDDVH3": [9],
    "BMI_CLASS": [6, 9],
    "INCDGHH": [9],
    "SDCDGIMM": [9],
    "GEOGPRV": [],
    "CCC_80": [9]
}

def apply_special_codes(df, special_codes):
    result = df.copy()

    for column, codes in special_codes.items():
        if column in result.columns:
            result[column] = result[column].replace(codes, np.nan)

    return result

clean_model_data = apply_special_codes(model_data, SPECIAL_CODES)

print("Missing values after special-code handling:")
print(clean_model_data[FEATURES].isna().sum())

Missing values after special-code handling:
DHHGAGE         0
DHH_SEX         0
EDDVH3       2276
BMI_CLASS    3144
INCDGHH       947
SDCDGIMM      835
GEOGPRV         0
CCC_80        494
dtype: int64


## Train / validation / test split

Same population, seed, and stratified split as A, B, C, and F.

In [6]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_val_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.20,
    stratify=model_data["target"],
    random_state=RANDOM_STATE
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.20,
    stratify=model_data.loc[train_val_idx, "target"],
    random_state=RANDOM_STATE
)

print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Training: 42394
Validation: 10599
Test: 13249


In [7]:
X_train = clean_model_data.loc[train_idx, FEATURES]
X_val = clean_model_data.loc[val_idx, FEATURES]
X_test = clean_model_data.loc[test_idx, FEATURES]

y_train = model_data.loc[train_idx, "target"]
y_val = model_data.loc[val_idx, "target"]
y_test = model_data.loc[test_idx, "target"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (42394, 8) (42394,)
Validation: (10599, 8) (10599,)
Test: (13249, 8) (13249,)


## Preprocessing

All 8 features are categorical, same as C. Fitted on training data only.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, FEATURES)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (42394, 32)
Processed validation shape: (10599, 32)
Processed test shape: (13249, 32)


## Logistic Regression

In [9]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

print("Model trained.")

Model trained.


In [10]:
train_prob = logistic_model.predict_proba(X_train_processed)[:, 1]
val_prob = logistic_model.predict_proba(X_val_processed)[:, 1]
test_prob = logistic_model.predict_proba(X_test_processed)[:, 1]

print("Predicted probabilities generated.")

Predicted probabilities generated.


## Threshold selection on validation

Same approach as C: sweep thresholds on validation, lock the one with the highest F1.

In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

thresholds = np.arange(0.10, 0.91, 0.01)

threshold_results = []

for threshold in thresholds:
    val_pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0),
        "accuracy": accuracy_score(y_val, val_pred)
    })

threshold_results = pd.DataFrame(threshold_results)

best_row = threshold_results.loc[threshold_results["f1"].idxmax()]

FINAL_THRESHOLD = float(best_row["threshold"])
VAL_F1 = float(best_row["f1"])

print("Selected validation threshold:", round(FINAL_THRESHOLD, 3))
print(best_row.round(4))

Selected validation threshold: 0.7
threshold    0.7000
precision    0.2737
recall       0.4755
f1           0.3474
accuracy     0.8384
Name: 60, dtype: float64


## Test evaluation

Threshold is locked. Test set is evaluated once.

In [12]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

final_test_pred = (test_prob >= FINAL_THRESHOLD).astype(int)

test_accuracy = accuracy_score(y_test, final_test_pred)
test_precision = precision_score(y_test, final_test_pred, zero_division=0)
test_recall = recall_score(y_test, final_test_pred, zero_division=0)
test_f1 = f1_score(y_test, final_test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, test_prob)
test_pr_auc = average_precision_score(y_test, test_prob)

print("Final Test Results")
print(f"Threshold: {FINAL_THRESHOLD:.3f}")
print(f"Accuracy:  {test_accuracy:.3f}")
print(f"Precision: {test_precision:.3f}")
print(f"Recall:    {test_recall:.3f}")
print(f"F1-score:  {test_f1:.3f}")
print(f"ROC-AUC:   {test_roc_auc:.3f}")
print(f"PR-AUC:    {test_pr_auc:.3f}")

cm = confusion_matrix(y_test, final_test_pred)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion matrix:")
print(cm)

Final Test Results
Threshold: 0.700
Accuracy:  0.832
Precision: 0.258
Recall:    0.457
F1-score:  0.330
ROC-AUC:   0.794
PR-AUC:    0.263

Confusion matrix:
[[10474  1576]
 [  651   548]]


In [13]:
from sklearn.metrics import log_loss

train_log_loss = log_loss(y_train, train_prob)
val_log_loss = log_loss(y_val, val_prob)
test_log_loss = log_loss(y_test, test_prob)

print("Log Loss")
print(f"Training:   {train_log_loss:.4f}")
print(f"Validation: {val_log_loss:.4f}")
print(f"Test:       {test_log_loss:.4f}")

Log Loss
Training:   0.5460
Validation: 0.5488
Test:       0.5456


## Failure analysis

Same style as C.

In [14]:
test_results = X_test.copy()

test_results["actual"] = y_test.values
test_results["predicted"] = final_test_pred
test_results["probability"] = test_prob

test_results["error_type"] = "Correct"
test_results.loc[
    (test_results["actual"] == 1) & (test_results["predicted"] == 0), "error_type"
] = "False Negative"
test_results.loc[
    (test_results["actual"] == 0) & (test_results["predicted"] == 1), "error_type"
] = "False Positive"

print(test_results["error_type"].value_counts())

error_type
Correct           11022
False Positive     1576
False Negative      651
Name: count, dtype: int64


In [15]:
ccc80_error_rates = (
    test_results
    .groupby(["CCC_80", "error_type"])
    .size()
    .unstack(fill_value=0)
)

ccc80_error_rates["Total"] = ccc80_error_rates.sum(axis=1)

for column in ["Correct", "False Negative", "False Positive"]:
    ccc80_error_rates[f"{column} Rate"] = ccc80_error_rates[column] / ccc80_error_rates["Total"]

print(ccc80_error_rates.round(3))

error_type  Correct  False Negative  False Positive  Total  Correct Rate  \
CCC_80                                                                     
1.0            1583             157            1497   3237         0.489   
2.0            9352             489              75   9916         0.943   

error_type  False Negative Rate  False Positive Rate  
CCC_80                                                
1.0                       0.049                0.462  
2.0                       0.049                0.008  


In [16]:
BORDERLINE_MARGIN = 0.05

false_negatives = test_results[test_results["error_type"] == "False Negative"]
false_positives = test_results[test_results["error_type"] == "False Positive"]

fn_borderline = (
    (false_negatives["probability"] >= FINAL_THRESHOLD - BORDERLINE_MARGIN) &
    (false_negatives["probability"] < FINAL_THRESHOLD)
)

fp_borderline = (
    (false_positives["probability"] >= FINAL_THRESHOLD) &
    (false_positives["probability"] <= FINAL_THRESHOLD + BORDERLINE_MARGIN)
)

print(
    f"False negatives within {BORDERLINE_MARGIN:.2f} of threshold: "
    f"{fn_borderline.sum()} / {len(false_negatives)} ({fn_borderline.mean():.1%})"
)

print(
    f"False positives within {BORDERLINE_MARGIN:.2f} of threshold: "
    f"{fp_borderline.sum()} / {len(false_positives)} ({fp_borderline.mean():.1%})"
)

False negatives within 0.05 of threshold: 114 / 651 (17.5%)
False positives within 0.05 of threshold: 408 / 1576 (25.9%)


## Save results

In [17]:
experiment_d1_results = pd.DataFrame([{
    "experiment": "D1_Core_Hypertension",
    "model": "Logistic Regression",
    "raw_feature_count": len(FEATURES),
    "processed_feature_count": X_train_processed.shape[1],
    "threshold": FINAL_THRESHOLD,
    "validation_f1": VAL_F1,
    "accuracy": test_accuracy,
    "precision": test_precision,
    "recall": test_recall,
    "f1": test_f1,
    "roc_auc": test_roc_auc,
    "pr_auc": test_pr_auc,
    "train_log_loss": train_log_loss,
    "validation_log_loss": val_log_loss,
    "test_log_loss": test_log_loss,
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
    "train_shape": str(X_train_processed.shape),
    "validation_shape": str(X_val_processed.shape),
    "test_shape": str(X_test_processed.shape)
}])

experiment_d1_results.to_csv("experiment_D1_results.csv", index=False)

print("Saved: experiment_D1_results.csv")
experiment_d1_results

Saved: experiment_D1_results.csv


,experiment,model,raw_feature_count,processed_feature_count,threshold,validation_f1,accuracy,precision,recall,f1,...,train_log_loss,validation_log_loss,test_log_loss,true_negatives,false_positives,false_negatives,true_positives,train_shape,validation_shape,test_shape
0,D1_Core_Hypertension,Logistic Regression,8,32,0.7,0.347429,0.831912,0.258004,0.457048,0.329822,...,0.545994,0.548768,0.545559,10474,1576,651,548,"(42394, 32)","(10599, 32)","(13249, 32)"


## Comparison with C

|  | C (7 features) | D1 (+ CCC_80) |
|---|---|---|
| Threshold | 0.660 | 0.700 |
| Accuracy | 0.788 | 0.832 |
| Precision | 0.218 | 0.258 |
| Recall | 0.519 | 0.457 |
| F1 | 0.307 | 0.330 |
| ROC-AUC | 0.774 | 0.794 |
| PR-AUC | 0.232 | 0.263 |

Adding CCC_80 improves accuracy, precision, ROC-AUC, PR-AUC, and F1, but lowers recall. The failure breakdown shows why: among respondents who report hypertension (CCC_80 = 1), the false positive rate is 46%, much higher than the 0.8% for those without hypertension. The model leans heavily on CCC_80 to classify diagnosed diabetes status, which helps overall discrimination but also produces more false alarms within that group.


## Results

8 raw features, 32 after one-hot encoding. Threshold 0.700 locked from validation F1.

Test set:
- Accuracy: 0.832, Precision: 0.258, Recall: 0.457, F1: 0.330
- ROC-AUC: 0.794, PR-AUC: 0.263
- Confusion matrix: TN 10474, FP 1576, FN 651, TP 548

Train/val/test log loss: 0.5460 / 0.5488 / 0.5456. Log loss is similar across the training, validation, and test sets, with no large train-to-test gap.

CCC_80 gives a real but modest lift over C on most metrics, at the cost of recall. See the comparison table above.
